# Module 10: Mixture of Experts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/10-mixture-of-experts/notebook.ipynb)

**GPU recommended:** No (toy MoE model trains on CPU in under 2 minutes).

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from typing import Optional, Tuple

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
print(f"Using device: {device}")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

---

## Part 1: Build an Expert FFN

Each expert is a standard feed-forward network: expand the dimension with a linear layer,
apply a GELU activation, then contract back to the original dimension. This is exactly
the FFN inside a Transformer block.

In [ ]:
class ExpertFFN(nn.Module):
    """A single expert: standard FFN with expand-activate-contract."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.linear_up = nn.Linear(d_model, d_ff)
        self.linear_down = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        hidden = F.gelu(self.linear_up(x))
        hidden = self.dropout(hidden)
        return self.linear_down(hidden)

In [ ]:
# Test the expert FFN
torch.manual_seed(42)

d_model = 64
d_ff = 256
expert = ExpertFFN(d_model, d_ff)

test_input = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
test_output = expert(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Expert parameters: {sum(p.numel() for p in expert.parameters()):,}")
print(f"  W_up:   {d_model} x {d_ff} = {d_model * d_ff:,}")
print(f"  W_down: {d_ff} x {d_model} = {d_ff * d_model:,}")
print(f"  Biases: {d_ff + d_model:,}")

---

## Part 2: Build a Router

The router (or gate network) takes a token representation and produces a probability
distribution over experts. We then select the top-k experts and renormalize their weights.

The router is a single linear layer followed by softmax:

$$g(x) = \text{Softmax}(W_g \cdot x)$$

Top-k selection picks the k experts with the highest gate values.

In [ ]:
class Router(nn.Module):
    """Top-k router with softmax gating."""

    def __init__(self, d_model: int, num_experts: int, top_k: int = 2,
                 jitter_noise: float = 0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.jitter_noise = jitter_noise
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            top_k_weights: (batch, seq_len, top_k) -- renormalized weights
            top_k_indices: (batch, seq_len, top_k) -- expert indices
            router_probs: (batch, seq_len, num_experts) -- full probability distribution
        """
        # Optional jitter noise during training for exploration
        if self.training and self.jitter_noise > 0:
            noise = torch.empty_like(x).uniform_(-self.jitter_noise, self.jitter_noise)
            x = x * (1.0 + noise)

        # Compute gate logits and probabilities
        logits = self.gate(x)  # (batch, seq_len, num_experts)
        router_probs = F.softmax(logits, dim=-1)

        # Select top-k experts
        top_k_weights, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)

        # Renormalize the top-k weights so they sum to 1
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        return top_k_weights, top_k_indices, router_probs

In [ ]:
# Test the router
torch.manual_seed(42)

num_experts = 8
top_k = 2
router = Router(d_model, num_experts, top_k)

test_tokens = torch.randn(1, 5, d_model)  # 5 tokens
weights, indices, probs = router(test_tokens)

print(f"Router probabilities shape: {probs.shape}")
print(f"Top-k weights shape: {weights.shape}")
print(f"Top-k indices shape: {indices.shape}")
print()
for token_idx in range(5):
    expert_ids = indices[0, token_idx].tolist()
    expert_weights = weights[0, token_idx].tolist()
    print(f"Token {token_idx}: experts {expert_ids}, "
          f"weights [{expert_weights[0]:.3f}, {expert_weights[1]:.3f}]")

---

## Part 3: MoE Layer

The MoE layer combines the router with a pool of expert FFNs. For each token:
1. The router selects top-k experts and their weights.
2. Each selected expert processes the token.
3. The output is the weighted sum of the selected experts' outputs.

$$\text{MoE}(x) = \sum_{i \in \text{TopK}} \hat{g}_i(x) \cdot \text{Expert}_i(x)$$

In [ ]:
class MoELayer(nn.Module):
    """Mixture of Experts layer: router + pool of expert FFNs."""

    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int = 2,
                 jitter_noise: float = 0.0, dropout: float = 0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        self.router = Router(d_model, num_experts, top_k, jitter_noise)
        self.experts = nn.ModuleList([
            ExpertFFN(d_model, d_ff, dropout) for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
            router_probs: (batch, seq_len, num_experts) for load balancing
        """
        batch_size, seq_len, d_model = x.shape
        top_k_weights, top_k_indices, router_probs = self.router(x)

        # Initialize output tensor
        output = torch.zeros_like(x)

        # Process each expert: gather tokens assigned to it, compute, scatter back
        # This loop is over experts (small, fixed number), not tokens
        for expert_idx in range(self.num_experts):
            # Create a mask for tokens that selected this expert (in any top-k slot)
            expert_mask = (top_k_indices == expert_idx)  # (batch, seq_len, top_k)

            if not expert_mask.any():
                continue

            # Get the corresponding weights for this expert
            expert_weights = (top_k_weights * expert_mask.float()).sum(dim=-1)  # (batch, seq_len)

            # Find which tokens need this expert
            token_mask = expert_weights > 0  # (batch, seq_len)

            if not token_mask.any():
                continue

            # Process all tokens through this expert (we could be more efficient
            # by only processing selected tokens, but this is clearer for learning)
            expert_output = self.experts[expert_idx](x)  # (batch, seq_len, d_model)

            # Weight the expert output and add to result
            output += expert_output * expert_weights.unsqueeze(-1)

        return output, router_probs

In [ ]:
# Test the MoE layer
torch.manual_seed(42)

moe_layer = MoELayer(d_model=64, d_ff=256, num_experts=8, top_k=2)

test_input = torch.randn(2, 10, 64)
moe_output, moe_probs = moe_layer(test_input)

total_params = sum(p.numel() for p in moe_layer.parameters())
expert_params = sum(p.numel() for p in moe_layer.experts.parameters())
router_params = sum(p.numel() for p in moe_layer.router.parameters())

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {moe_output.shape}")
print(f"Router probs shape: {moe_probs.shape}")
print()
print(f"Total parameters:   {total_params:,}")
print(f"  Router parameters:  {router_params:,}")
print(f"  Expert parameters:  {expert_params:,} ({expert_params // 8:,} per expert)")
print(f"  Active per token:   {router_params + expert_params // 8 * 2:,} (router + 2 experts)")
print(f"  Capacity/compute ratio: {total_params / (router_params + expert_params // 8 * 2):.1f}x")

---

## Part 4: Visualize Routing Decisions

Let's see how the (untrained) router distributes tokens across experts. We'll look at
the full probability distribution and which experts are selected.

In [ ]:
torch.manual_seed(42)

# Generate diverse inputs to visualize routing
num_tokens = 100
diverse_inputs = torch.randn(1, num_tokens, d_model)

moe_layer.eval()
with torch.no_grad():
    _, _, routing_probs = moe_layer.router(diverse_inputs)
    top_weights, top_indices, _ = moe_layer.router(diverse_inputs)

routing_probs_np = routing_probs[0].numpy()  # (num_tokens, num_experts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap of router probabilities for first 30 tokens
im = axes[0].imshow(routing_probs_np[:30].T, aspect="auto", cmap="YlOrRd",
                     vmin=0, vmax=routing_probs_np[:30].max())
axes[0].set_xlabel("Token index")
axes[0].set_ylabel("Expert index")
axes[0].set_title("Router probability distribution (first 30 tokens)")
axes[0].set_yticks(range(num_experts))
plt.colorbar(im, ax=axes[0], label="Probability")

# Right: histogram of expert selection frequency
expert_counts = np.zeros(num_experts)
for token_idx in range(num_tokens):
    for k_idx in range(top_k):
        expert_id = top_indices[0, token_idx, k_idx].item()
        expert_counts[expert_id] += 1

colors = plt.cm.Set2(np.linspace(0, 1, num_experts))
bars = axes[1].bar(range(num_experts), expert_counts, color=colors, edgecolor="gray")
axes[1].axhline(y=num_tokens * top_k / num_experts, color="red", linestyle="--",
                 label=f"Perfectly balanced ({num_tokens * top_k / num_experts:.0f})")
axes[1].set_xlabel("Expert index")
axes[1].set_ylabel("Number of tokens routed")
axes[1].set_title("Expert utilization (untrained router)")
axes[1].set_xticks(range(num_experts))
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Part 5: Load Balancing Loss

Without a load balancing loss, the router tends to collapse: it sends most tokens to
a small number of "favorite" experts. The auxiliary loss encourages uniform utilization.

$$\mathcal{L}_{\text{balance}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $p_i$ is the average
router probability for expert $i$.

In [ ]:
def load_balancing_loss(router_probs: torch.Tensor,
                        top_k_indices: torch.Tensor,
                        num_experts: int,
                        alpha: float = 0.01) -> torch.Tensor:
    """
    Compute the auxiliary load balancing loss.

    Args:
        router_probs: (batch, seq_len, num_experts) full probability distribution
        top_k_indices: (batch, seq_len, top_k) selected expert indices
        num_experts: total number of experts
        alpha: loss weight

    Returns:
        Scalar loss value
    """
    batch_size, seq_len, _ = router_probs.shape
    total_tokens = batch_size * seq_len

    # f_i: fraction of tokens routed to each expert
    # Create one-hot masks for selected experts and sum
    expert_mask = F.one_hot(top_k_indices, num_experts).float()  # (batch, seq_len, top_k, num_experts)
    expert_mask = expert_mask.sum(dim=2)  # (batch, seq_len, num_experts)
    tokens_per_expert = expert_mask.sum(dim=(0, 1))  # (num_experts,)
    fraction_per_expert = tokens_per_expert / total_tokens  # f_i

    # p_i: average router probability for each expert
    avg_prob_per_expert = router_probs.mean(dim=(0, 1))  # (num_experts,)

    # Load balancing loss
    loss = alpha * num_experts * (fraction_per_expert * avg_prob_per_expert).sum()

    return loss

In [ ]:
# Demonstrate: compute the load balancing loss for our test case
torch.manual_seed(42)

test_input = torch.randn(4, 20, d_model)  # batch=4, seq_len=20
moe_layer.train()
_, router_probs = moe_layer(test_input)
_, top_k_idx, _ = moe_layer.router(test_input)

lb_loss = load_balancing_loss(router_probs, top_k_idx, num_experts, alpha=0.01)
print(f"Load balancing loss: {lb_loss.item():.6f}")
print(f"Perfectly balanced would give: {0.01:.6f}")
print(f"Ratio to perfect: {lb_loss.item() / 0.01:.2f}x")

### Demonstrating Expert Collapse

Let's train a small MoE on a simple task -- first *without* load balancing loss, then
*with* it -- to see the difference in expert utilization.

In [ ]:
def generate_classification_data(num_samples: int, d_model: int, num_classes: int):
    """
    Generate a simple classification dataset: each class has a distinct
    cluster center in d_model space.
    """
    torch.manual_seed(42)
    centers = torch.randn(num_classes, d_model) * 3.0
    labels = torch.randint(0, num_classes, (num_samples,))
    data = centers[labels] + torch.randn(num_samples, d_model) * 0.5
    return data, labels


def train_moe_classifier(use_load_balancing: bool, num_epochs: int = 150,
                          alpha: float = 0.1):
    """Train a simple MoE model on classification, optionally with load balancing."""
    torch.manual_seed(42)

    num_classes = 6
    train_data, train_labels = generate_classification_data(500, d_model, num_classes)

    # Simple model: MoE layer followed by a classification head
    moe = MoELayer(d_model=d_model, d_ff=128, num_experts=8, top_k=2)
    classifier_head = nn.Linear(d_model, num_classes)

    params = list(moe.parameters()) + list(classifier_head.parameters())
    optimizer = torch.optim.Adam(params, lr=1e-3)

    losses = []
    expert_usage_history = []

    for epoch in range(num_epochs):
        moe.train()
        # Treat each sample as a sequence of length 1
        x = train_data.unsqueeze(1)  # (500, 1, d_model)
        moe_out, router_probs = moe(x)
        logits = classifier_head(moe_out.squeeze(1))

        # Task loss
        task_loss = F.cross_entropy(logits, train_labels)

        # Load balancing loss
        _, top_k_idx, _ = moe.router(x)
        if use_load_balancing:
            lb_loss = load_balancing_loss(router_probs, top_k_idx, 8, alpha=alpha)
            total_loss = task_loss + lb_loss
        else:
            total_loss = task_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        losses.append(task_loss.item())

        # Track expert usage
        if (epoch + 1) % 10 == 0:
            with torch.no_grad():
                expert_counts = torch.zeros(8)
                for k_slot in range(2):
                    for eidx in range(8):
                        expert_counts[eidx] += (top_k_idx[:, :, k_slot] == eidx).sum().item()
                expert_usage_history.append(expert_counts.numpy().copy())

    return losses, expert_usage_history

In [ ]:
# Train without and with load balancing
losses_no_lb, usage_no_lb = train_moe_classifier(use_load_balancing=False)
losses_with_lb, usage_with_lb = train_moe_classifier(use_load_balancing=True, alpha=0.1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top row: loss curves
axes[0, 0].plot(losses_no_lb, color="#e74c3c", alpha=0.8, label="Without LB")
axes[0, 0].plot(losses_with_lb, color="#2ecc71", alpha=0.8, label="With LB")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Task loss (cross-entropy)")
axes[0, 0].set_title("Training loss comparison")
axes[0, 0].legend()

# Top right: final expert usage without LB
final_usage_no_lb = usage_no_lb[-1]
total_routes = final_usage_no_lb.sum()
pct_no_lb = final_usage_no_lb / total_routes * 100
colors_no_lb = ["#e74c3c" if p > 20 else "#bdc3c7" for p in pct_no_lb]
axes[0, 1].bar(range(8), pct_no_lb, color=colors_no_lb, edgecolor="gray")
axes[0, 1].axhline(y=12.5, color="blue", linestyle="--", alpha=0.5,
                     label="Ideal (12.5%)")
axes[0, 1].set_xlabel("Expert index")
axes[0, 1].set_ylabel("Token share (%)")
axes[0, 1].set_title("Expert utilization -- WITHOUT load balancing")
axes[0, 1].set_xticks(range(8))
axes[0, 1].legend()

# Bottom left: expert usage over time without LB
usage_no_lb_arr = np.array(usage_no_lb)
usage_no_lb_pct = usage_no_lb_arr / usage_no_lb_arr.sum(axis=1, keepdims=True) * 100
for eidx in range(8):
    axes[1, 0].plot(range(10, 151, 10), usage_no_lb_pct[:, eidx],
                     label=f"Expert {eidx}", marker="o", markersize=3)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Token share (%)")
axes[1, 0].set_title("Expert utilization over training -- WITHOUT LB")
axes[1, 0].legend(fontsize=8, ncol=2)

# Bottom right: final expert usage with LB
final_usage_lb = usage_with_lb[-1]
pct_lb = final_usage_lb / final_usage_lb.sum() * 100
colors_lb = ["#2ecc71" for _ in pct_lb]
axes[1, 1].bar(range(8), pct_lb, color=colors_lb, edgecolor="gray")
axes[1, 1].axhline(y=12.5, color="blue", linestyle="--", alpha=0.5,
                     label="Ideal (12.5%)")
axes[1, 1].set_xlabel("Expert index")
axes[1, 1].set_ylabel("Token share (%)")
axes[1, 1].set_title("Expert utilization -- WITH load balancing")
axes[1, 1].set_xticks(range(8))
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---

## Part 6: Full MoE Transformer

Now we build a complete Transformer where the FFN sub-layer is replaced by our MoE layer.
We'll train it on a digit reversal task: given a sequence of digits, output them in
reverse order (same task as Module 02, allowing direct comparison).

For example: `[3, 1, 4, 1, 5]` -> `[5, 1, 4, 1, 3]`

In [ ]:
class MoETransformerBlock(nn.Module):
    """A single Transformer block with MoE replacing the FFN."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 num_experts: int, top_k: int, dropout: float = 0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads,
                                                dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, d_ff, num_experts, top_k,
                             jitter_noise=0.01, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        # Self-attention with residual + layer norm
        attn_out, _ = self.attention(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))

        # MoE FFN with residual + layer norm
        moe_out, router_probs = self.moe(x)
        x = self.norm2(x + self.dropout(moe_out))

        return x, router_probs

In [ ]:
class MoETransformer(nn.Module):
    """Encoder-only MoE Transformer for sequence-to-sequence tasks."""

    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 d_ff: int, num_layers: int, num_experts: int, top_k: int,
                 max_seq_len: int = 128, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_experts = num_experts

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        self.layers = nn.ModuleList([
            MoETransformerBlock(d_model, num_heads, d_ff, num_experts, top_k, dropout)
            for _ in range(num_layers)
        ])

        self.output_head = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, list]:
        batch_size, seq_len = x.shape

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        x = self.dropout(self.embedding(x) + self.pos_embedding(positions))

        all_router_probs = []
        for layer in self.layers:
            x, router_probs = layer(x, mask)
            all_router_probs.append(router_probs)

        logits = self.output_head(x)
        return logits, all_router_probs

In [ ]:
def generate_reversal_data(num_samples: int, seq_len: int, vocab_size: int):
    """
    Generate digit reversal data.
    Input: random digit sequences. Target: reversed sequences.
    """
    torch.manual_seed(42)
    inputs = torch.randint(0, vocab_size, (num_samples, seq_len))
    targets = inputs.flip(dims=[1])
    return inputs, targets


# Dataset parameters
vocab_size = 10  # digits 0-9
seq_len = 8
num_train = 2000
num_test = 500

train_inputs, train_targets = generate_reversal_data(num_train, seq_len, vocab_size)
test_inputs, test_targets = generate_reversal_data(num_test, seq_len, vocab_size)

print(f"Training data: {train_inputs.shape}")
print(f"Test data:     {test_inputs.shape}")
print()
print("Example:")
print(f"  Input:  {train_inputs[0].tolist()}")
print(f"  Target: {train_targets[0].tolist()}")

In [ ]:
def train_model(model, train_inputs, train_targets, num_experts,
                num_epochs=80, batch_size=64, lr=3e-4,
                use_load_balancing=True, lb_alpha=0.01):
    """
    Train a model on the reversal task. Returns loss history and
    per-epoch expert utilization stats.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_samples = train_inputs.shape[0]

    loss_history = []
    accuracy_history = []
    expert_utilization_history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        epoch_expert_counts = torch.zeros(num_experts)

        # Shuffle data
        perm = torch.randperm(num_samples)
        shuffled_inputs = train_inputs[perm]
        shuffled_targets = train_targets[perm]

        for batch_start in range(0, num_samples, batch_size):
            batch_end = min(batch_start + batch_size, num_samples)
            batch_x = shuffled_inputs[batch_start:batch_end]
            batch_y = shuffled_targets[batch_start:batch_end]

            logits, all_router_probs = model(batch_x)

            # Task loss: cross-entropy over all positions
            task_loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                        batch_y.view(-1))

            # Load balancing loss across all layers
            total_loss = task_loss
            if use_load_balancing and num_experts > 1:
                for layer_idx, layer in enumerate(model.layers):
                    with torch.no_grad():
                        _, top_k_idx, _ = layer.moe.router(batch_x.float().unsqueeze(-1).expand(
                            -1, -1, model.d_model)) if False else (None, None, None)

                # Recompute routing to get indices
                x_emb = model.dropout(model.embedding(batch_x) +
                                      model.pos_embedding(
                                          torch.arange(batch_x.size(1)).unsqueeze(0).expand(
                                              batch_x.size(0), -1)))
                temp_x = x_emb
                for layer in model.layers:
                    attn_out, _ = layer.attention(temp_x, temp_x, temp_x)
                    temp_x_normed = layer.norm1(temp_x + attn_out)
                    _, top_k_idx, _ = layer.moe.router(temp_x_normed)
                    lb = load_balancing_loss(
                        all_router_probs[0],  # use the stored probs
                        top_k_idx, num_experts, alpha=lb_alpha
                    )
                    total_loss = total_loss + lb
                    moe_out, _ = layer.moe(temp_x_normed)
                    temp_x = layer.norm2(temp_x_normed + moe_out)

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            epoch_loss += task_loss.item() * (batch_end - batch_start)

            # Accuracy
            preds = logits.argmax(dim=-1)
            epoch_correct += (preds == batch_y).sum().item()
            epoch_total += batch_y.numel()

        avg_loss = epoch_loss / num_samples
        accuracy = epoch_correct / epoch_total
        loss_history.append(avg_loss)
        accuracy_history.append(accuracy)

        # Track expert utilization
        if num_experts > 1:
            model.eval()
            with torch.no_grad():
                _, all_rp = model(train_inputs[:200])
                # Get routing decisions from first layer
                x_temp = model.dropout(model.embedding(train_inputs[:200]) +
                                        model.pos_embedding(
                                            torch.arange(seq_len).unsqueeze(0).expand(200, -1)))
                attn_out, _ = model.layers[0].attention(x_temp, x_temp, x_temp)
                x_normed = model.layers[0].norm1(x_temp + attn_out)
                _, top_k_idx, _ = model.layers[0].moe.router(x_normed)
                counts = torch.zeros(num_experts)
                for eidx in range(num_experts):
                    counts[eidx] = (top_k_idx == eidx).sum().item()
                expert_utilization_history.append(counts.numpy().copy())

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

    return loss_history, accuracy_history, expert_utilization_history

In [ ]:
# Build and train the MoE Transformer
torch.manual_seed(42)

moe_config = {
    "vocab_size": vocab_size,
    "d_model": 64,
    "num_heads": 4,
    "d_ff": 128,
    "num_layers": 2,
    "num_experts": 8,
    "top_k": 2,
    "max_seq_len": seq_len,
    "dropout": 0.1,
}

moe_transformer = MoETransformer(**moe_config)

total_params = sum(p.numel() for p in moe_transformer.parameters())
print(f"MoE Transformer total parameters: {total_params:,}")
print(f"Number of experts: {moe_config['num_experts']}, Top-k: {moe_config['top_k']}")
print()

moe_losses, moe_accuracies, moe_expert_util = train_model(
    moe_transformer, train_inputs, train_targets,
    num_experts=8, num_epochs=80, lr=3e-4,
    use_load_balancing=True, lb_alpha=0.01
)

In [ ]:
# Evaluate on test set
moe_transformer.eval()
with torch.no_grad():
    test_logits, _ = moe_transformer(test_inputs)
    test_preds = test_logits.argmax(dim=-1)
    test_accuracy = (test_preds == test_targets).float().mean().item()

    # Per-position accuracy
    position_accuracy = (test_preds == test_targets).float().mean(dim=0)

print(f"Test accuracy (token-level): {test_accuracy:.4f}")
print(f"Per-position accuracy: {[f'{a:.3f}' for a in position_accuracy.tolist()]}")

# Show some examples
print("\nExamples:")
for idx in range(5):
    inp = test_inputs[idx].tolist()
    tgt = test_targets[idx].tolist()
    pred = test_preds[idx].tolist()
    correct = "correct" if pred == tgt else "wrong"
    print(f"  Input: {inp}  Target: {tgt}  Predicted: {pred}  [{correct}]")

---

## Part 7: Expert Specialization Analysis

After training, we can analyze which types of inputs each expert tends to handle.
We look at: which digits activate which experts, and overall utilization histograms.

In [ ]:
def analyze_expert_specialization(model, inputs, seq_len, num_experts, vocab_size):
    """
    Analyze which experts handle which digit values.
    Returns a (num_experts, vocab_size) matrix of routing counts.
    """
    model.eval()
    digit_expert_counts = np.zeros((num_experts, vocab_size))
    position_expert_counts = np.zeros((num_experts, seq_len))

    with torch.no_grad():
        # Get routing decisions from the first layer
        x = model.dropout(model.embedding(inputs) +
                           model.pos_embedding(
                               torch.arange(seq_len).unsqueeze(0).expand(inputs.size(0), -1)))
        attn_out, _ = model.layers[0].attention(x, x, x)
        x_normed = model.layers[0].norm1(x + attn_out)
        _, top_k_idx, router_probs = model.layers[0].moe.router(x_normed)

        # Count which experts are selected for each digit value
        for sample_idx in range(inputs.size(0)):
            for pos_idx in range(seq_len):
                digit = inputs[sample_idx, pos_idx].item()
                for k_slot in range(top_k_idx.size(-1)):
                    expert_id = top_k_idx[sample_idx, pos_idx, k_slot].item()
                    digit_expert_counts[expert_id, digit] += 1
                    position_expert_counts[expert_id, pos_idx] += 1

    return digit_expert_counts, position_expert_counts, router_probs.numpy()

In [ ]:
digit_expert, pos_expert, final_probs = analyze_expert_specialization(
    moe_transformer, train_inputs[:500], seq_len, 8, vocab_size
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: digit-expert heatmap
# Normalize per digit (column) to show preference
digit_expert_norm = digit_expert / (digit_expert.sum(axis=0, keepdims=True) + 1e-8)
im1 = axes[0].imshow(digit_expert_norm, aspect="auto", cmap="Blues")
axes[0].set_xlabel("Digit value")
axes[0].set_ylabel("Expert index")
axes[0].set_title("Expert preference by digit value")
axes[0].set_xticks(range(vocab_size))
axes[0].set_yticks(range(8))
plt.colorbar(im1, ax=axes[0], label="Routing fraction")

# Middle: position-expert heatmap
pos_expert_norm = pos_expert / (pos_expert.sum(axis=0, keepdims=True) + 1e-8)
im2 = axes[1].imshow(pos_expert_norm, aspect="auto", cmap="Greens")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Expert index")
axes[1].set_title("Expert preference by position")
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(8))
plt.colorbar(im2, ax=axes[1], label="Routing fraction")

# Right: overall expert utilization histogram
total_per_expert = digit_expert.sum(axis=1)
total_routes = total_per_expert.sum()
utilization_pct = total_per_expert / total_routes * 100
colors = plt.cm.Set2(np.linspace(0, 1, 8))
axes[2].bar(range(8), utilization_pct, color=colors, edgecolor="gray")
axes[2].axhline(y=12.5, color="red", linestyle="--", alpha=0.6, label="Ideal (12.5%)")
axes[2].set_xlabel("Expert index")
axes[2].set_ylabel("Utilization (%)")
axes[2].set_title("Expert utilization histogram")
axes[2].set_xticks(range(8))
axes[2].legend()

plt.tight_layout()
plt.show()

# Print utilization statistics
print("Expert utilization:")
for eidx in range(8):
    print(f"  Expert {eidx}: {utilization_pct[eidx]:.1f}% of tokens")
print(f"\nStd dev of utilization: {np.std(utilization_pct):.2f}%")
print(f"Coefficient of variation: {np.std(utilization_pct) / np.mean(utilization_pct):.3f}")

---

## Part 8: Capacity Factor Experiment

The capacity factor limits how many tokens an expert can handle. We implement a version
of the MoE layer with token dropping and vary the capacity factor to observe its effect.

In [ ]:
class MoELayerWithCapacity(nn.Module):
    """MoE layer with capacity factor and token dropping."""

    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int = 2,
                 capacity_factor: float = 1.25, dropout: float = 0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor

        self.router = Router(d_model, num_experts, top_k)
        self.experts = nn.ModuleList([
            ExpertFFN(d_model, d_ff, dropout) for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, int]:
        batch_size, seq_len, d_model_dim = x.shape
        total_tokens = batch_size * seq_len

        # Compute capacity: max tokens per expert
        capacity = int(self.capacity_factor * total_tokens * self.top_k / self.num_experts)
        capacity = max(capacity, 1)

        top_k_weights, top_k_indices, router_probs = self.router(x)

        output = torch.zeros_like(x)
        total_dropped = 0

        for expert_idx in range(self.num_experts):
            expert_mask = (top_k_indices == expert_idx)  # (batch, seq_len, top_k)
            expert_weights = (top_k_weights * expert_mask.float()).sum(dim=-1)

            # Find tokens routed to this expert
            token_mask = expert_weights > 0
            num_routed = token_mask.sum().item()

            if num_routed == 0:
                continue

            # Apply capacity constraint: if too many tokens, drop the excess
            if num_routed > capacity:
                # Keep only the top-capacity tokens by gate weight
                flat_weights = expert_weights.view(-1)
                flat_mask = token_mask.view(-1)
                active_indices = flat_mask.nonzero(as_tuple=True)[0]
                active_weights = flat_weights[active_indices]
                _, keep_order = active_weights.topk(capacity)
                keep_indices = active_indices[keep_order]

                new_mask = torch.zeros_like(flat_mask)
                new_mask[keep_indices] = True
                token_mask = new_mask.view(batch_size, seq_len)
                expert_weights = expert_weights * token_mask.float()

                total_dropped += num_routed - capacity

            expert_output = self.experts[expert_idx](x)
            output += expert_output * expert_weights.unsqueeze(-1)

        return output, router_probs, total_dropped

In [ ]:
def run_capacity_experiment(capacity_factors):
    """Train MoE with different capacity factors and measure results."""
    results = {}

    for cf in capacity_factors:
        torch.manual_seed(42)
        print(f"\nCapacity factor: {cf}")

        moe_cap = MoELayerWithCapacity(
            d_model=64, d_ff=128, num_experts=8, top_k=2,
            capacity_factor=cf
        )
        classifier = nn.Linear(64, vocab_size)

        params = list(moe_cap.parameters()) + list(classifier.parameters())
        optimizer = torch.optim.Adam(params, lr=1e-3)

        losses = []
        drop_counts = []

        # Simple training: predict the first digit from the sequence mean
        for epoch in range(100):
            moe_cap.train()
            x = train_inputs[:300].unsqueeze(1).float()  # use digit values as features
            # Create a simple feature representation
            x_embed = torch.zeros(300, seq_len, 64)
            for pos in range(seq_len):
                x_embed[:, pos, train_inputs[:300, pos] * 6] = 1.0  # one-hot-ish
                x_embed[:, pos, pos + 50] = 1.0  # positional signal

            moe_out, rp, dropped = moe_cap(x_embed)
            pooled = moe_out.mean(dim=1)
            logits = classifier(pooled)
            target = train_targets[:300, 0]  # predict first digit of reversed seq

            task_loss = F.cross_entropy(logits, target)
            _, top_k_idx, _ = moe_cap.router(x_embed)
            lb = load_balancing_loss(rp, top_k_idx, 8, alpha=0.01)
            total_loss = task_loss + lb

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            losses.append(task_loss.item())
            drop_counts.append(dropped)

        # Final expert utilization
        moe_cap.eval()
        with torch.no_grad():
            _, _, final_dropped = moe_cap(x_embed)
            _, final_indices, _ = moe_cap.router(x_embed)
            expert_counts = torch.zeros(8)
            for eidx in range(8):
                expert_counts[eidx] = (final_indices == eidx).sum().item()

        results[cf] = {
            "losses": losses,
            "drop_counts": drop_counts,
            "expert_counts": expert_counts.numpy(),
            "final_loss": losses[-1],
            "total_dropped": sum(drop_counts[-10:]),
        }
        print(f"  Final loss: {losses[-1]:.4f}, "
              f"Avg tokens dropped (last 10 epochs): {sum(drop_counts[-10:])/10:.0f}")

    return results


capacity_factors = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
cap_results = run_capacity_experiment(capacity_factors)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: loss curves for different capacity factors
cmap = plt.cm.viridis(np.linspace(0, 1, len(capacity_factors)))
for idx, cf in enumerate(capacity_factors):
    axes[0].plot(cap_results[cf]["losses"], color=cmap[idx], alpha=0.8,
                 label=f"CF={cf}")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Task loss")
axes[0].set_title("Training loss by capacity factor")
axes[0].legend(fontsize=9)

# Middle: final loss vs capacity factor
final_losses = [cap_results[cf]["final_loss"] for cf in capacity_factors]
axes[1].plot(capacity_factors, final_losses, "o-", color="#2ecc71",
              markersize=8, linewidth=2)
axes[1].set_xlabel("Capacity factor")
axes[1].set_ylabel("Final training loss")
axes[1].set_title("Final loss vs capacity factor")

# Right: tokens dropped vs capacity factor
avg_dropped = [cap_results[cf]["total_dropped"] / 10 for cf in capacity_factors]
axes[2].bar(range(len(capacity_factors)), avg_dropped,
             tick_label=[str(cf) for cf in capacity_factors],
             color="#e74c3c", alpha=0.7, edgecolor="gray")
axes[2].set_xlabel("Capacity factor")
axes[2].set_ylabel("Avg tokens dropped per epoch")
axes[2].set_title("Token dropping by capacity factor")

plt.tight_layout()
plt.show()

---

## Part 9: Compare MoE vs Dense

The key claim of MoE is: more quality per FLOP than dense models.
We train a dense Transformer with the same number of *active* parameters
as our MoE model and compare loss curves.

- MoE: 8 experts, top-2, d_ff=128 per expert. Active FFN params per token = 2 experts.
- Dense: single FFN with d_ff=256 (matched to 2 experts' capacity).

In [ ]:
class DenseTransformerBlock(nn.Module):
    """Standard Transformer block with a dense FFN (no MoE)."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 dropout: float = 0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads,
                                                dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = ExpertFFN(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        attn_out, _ = self.attention(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x


class DenseTransformer(nn.Module):
    """Standard dense Transformer for comparison with MoE."""

    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 d_ff: int, num_layers: int, max_seq_len: int = 128,
                 dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        self.layers = nn.ModuleList([
            DenseTransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.output_head = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        x = self.dropout(self.embedding(x) + self.pos_embedding(positions))

        for layer in self.layers:
            x = layer(x, mask)

        return self.output_head(x)

In [ ]:
def train_dense_model(model, train_inputs, train_targets,
                       num_epochs=80, batch_size=64, lr=3e-4):
    """Train a dense Transformer on the reversal task."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_samples = train_inputs.shape[0]

    loss_history = []
    accuracy_history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        perm = torch.randperm(num_samples)
        shuffled_inputs = train_inputs[perm]
        shuffled_targets = train_targets[perm]

        for batch_start in range(0, num_samples, batch_size):
            batch_end = min(batch_start + batch_size, num_samples)
            batch_x = shuffled_inputs[batch_start:batch_end]
            batch_y = shuffled_targets[batch_start:batch_end]

            logits = model(batch_x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    batch_y.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * (batch_end - batch_start)
            preds = logits.argmax(dim=-1)
            epoch_correct += (preds == batch_y).sum().item()
            epoch_total += batch_y.numel()

        avg_loss = epoch_loss / num_samples
        accuracy = epoch_correct / epoch_total
        loss_history.append(avg_loss)
        accuracy_history.append(accuracy)

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

    return loss_history, accuracy_history

In [ ]:
# Dense model: match the ACTIVE parameters of the MoE model
# MoE: d_ff=128 per expert, top-2 of 8 -> active FFN d_ff = 256
torch.manual_seed(42)

dense_model = DenseTransformer(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    d_ff=256,  # 2 * 128 to match active params of top-2 MoE
    num_layers=2,
    max_seq_len=seq_len,
    dropout=0.1,
)

dense_params = sum(p.numel() for p in dense_model.parameters())
moe_total_params = sum(p.numel() for p in moe_transformer.parameters())

print(f"Dense model parameters:     {dense_params:,}")
print(f"MoE model total parameters: {moe_total_params:,}")
print(f"MoE/Dense param ratio:      {moe_total_params / dense_params:.2f}x")
print()
print("Training dense model...")
dense_losses, dense_accuracies = train_dense_model(
    dense_model, train_inputs, train_targets,
    num_epochs=80, lr=3e-4
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
axes[0].plot(moe_losses, color="#e74c3c", alpha=0.8, linewidth=2, label="MoE (8 experts, top-2)")
axes[0].plot(dense_losses, color="#3498db", alpha=0.8, linewidth=2, label="Dense (matched active params)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training loss")
axes[0].set_title("MoE vs Dense: Training Loss")
axes[0].legend()

# Accuracy comparison
axes[1].plot(moe_accuracies, color="#e74c3c", alpha=0.8, linewidth=2, label="MoE (8 experts, top-2)")
axes[1].plot(dense_accuracies, color="#3498db", alpha=0.8, linewidth=2, label="Dense (matched active params)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training accuracy")
axes[1].set_title("MoE vs Dense: Training Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

# Test both models
moe_transformer.eval()
dense_model.eval()

with torch.no_grad():
    moe_test_logits, _ = moe_transformer(test_inputs)
    moe_test_acc = (moe_test_logits.argmax(-1) == test_targets).float().mean().item()

    dense_test_logits = dense_model(test_inputs)
    dense_test_acc = (dense_test_logits.argmax(-1) == test_targets).float().mean().item()

print(f"\nTest accuracy:")
print(f"  MoE   ({moe_total_params:,} total params, ~{moe_total_params//4:,} active): {moe_test_acc:.4f}")
print(f"  Dense ({dense_params:,} params):                                  {dense_test_acc:.4f}")

In [ ]:
# Summary: parameter efficiency comparison
fig, ax = plt.subplots(figsize=(8, 5))

models = ["Dense\n(matched active)", "MoE 8x\n(top-2)"]
total_params_list = [dense_params, moe_total_params]
test_accs = [dense_test_acc, moe_test_acc]

bar_width = 0.35
x_pos = np.arange(len(models))

bars1 = ax.bar(x_pos - bar_width/2, [p / 1000 for p in total_params_list],
                bar_width, label="Total params (K)", color="#3498db", alpha=0.7)
ax.set_ylabel("Parameters (thousands)")
ax.set_xticks(x_pos)
ax.set_xticklabels(models)
ax.set_title("MoE vs Dense: Parameter Efficiency")

ax2 = ax.twinx()
bars2 = ax2.bar(x_pos + bar_width/2, [a * 100 for a in test_accs],
                 bar_width, label="Test accuracy (%)", color="#e74c3c", alpha=0.7)
ax2.set_ylabel("Test accuracy (%)")
ax2.set_ylim(0, 105)

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

print("Key takeaway:")
print(f"  MoE has {moe_total_params / dense_params:.1f}x more total parameters")
print(f"  but only activates ~{100 * 2/8:.0f}% of them per token (top-2 of 8 experts).")
print(f"  The extra capacity allows MoE to store more specialized knowledge,")
print(f"  potentially achieving better quality per FLOP on larger-scale tasks.")

---

## Summary

In this notebook we built Mixture of Experts from scratch:

1. **Expert FFN** -- standard feed-forward network used as each expert.
2. **Router** -- softmax gate with top-k selection.
3. **MoE Layer** -- routes tokens to selected experts and combines outputs.
4. **Routing visualization** -- saw how tokens are distributed across experts.
5. **Load balancing** -- the auxiliary loss that prevents expert collapse.
6. **MoE Transformer** -- full model with MoE replacing FFN layers.
7. **Expert specialization** -- analyzed what each expert learns to handle.
8. **Capacity factor** -- explored the tradeoff between capacity and dropping.
9. **MoE vs Dense** -- compared efficiency at matched active parameter counts.

The core insight: MoE decouples model capacity from compute cost, enabling
models that store more knowledge while using less computation per token.